In [2]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/gene_mapping_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")

Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [ ]:
n_workers <- 20
mod_def <- "TopModPosBC"

max_set_size <- 200
min_set_size <- 3

## Prep marker genes

### Claude marker genes

In [4]:
set_source <- "Claude_marker_genes"

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

### MO marker genes

In [ ]:
load("/mnt/lareaulab/reliscu/data/gene_sets/Oldham/MO_sets.RData")

mask <- !MO_legend$Category %in% "Disease"
marker_genes_list <- MO_sets[mask]
names(marker_genes_list) <- MO_legend$SetName[mask]

set_sizes <- lengths(marker_genes_list)
mask <- (set_sizes < max_set_size) & (set_sizes > min_set_size)
marker_genes_list <- marker_genes_list[mask]

print(length(marker_genes_list))

set_source <- paste0("MO_", length(marker_genes_list), "sets")

### Broad

In [ ]:
library(msigdbr)

gs_GOBP <- msigdbr(collection="C5", subcollection="GO:BP")

gs_CTS <- msigdbr(collection="C8")
gs_CTS <- gs_CTS[gs_CTS$gs_geoid == "GSE104276",]

gs_GOBP_list <- tapply(gs_GOBP$gene_symbol, INDEX=gs_GOBP$gs_name, FUN=list)
gs_CTS_list <- tapply(gs_CTS$gene_symbol, INDEX=gs_CTS$gs_name, FUN=list)

gs_list <- c(gs_GOBP_list, gs_CTS_list)

In [ ]:
set_sizes <- lengths(gs_list)
mask <- (set_sizes < max_set_size) & (set_sizes > min_set_size)
gs_list <- gs_list[mask]

print(length(gs_list))

In [ ]:
set_source <- paste0("MSigDB_", length(gs_list), "sets")

marker_genes_list <- gs_list

## Get enrichments

In [ ]:
network_dir <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned_mergeParam0.95_subsetCutoff8_Modules"

In [ ]:
enrichments_df <- get_module_enrichments_par(network_dir, marker_genes_list, mod_def, n_workers)

# enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

write.csv(enrichments_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)

# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)
    
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

# get_mod_vs_DE_genes(top_mods_df, marker_genes_list, mod_def)

write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments_top_Qval_mods.csv"), row.names=FALSE, quote=FALSE)